In [10]:

!pip install diffusers transformers accelerate

In [11]:
from diffusers.pipelines.pipeline_loading_utils import variant_compatible_siblings
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepInverseScheduler
from diffusers.utils import export_to_video

pipe=DiffusionPipeline.from_pretrained("damo-vilab/text-to-video-ms-1.7b", torch_dtype=torch.float16, variant="fp16")
pipe.scheduler=DPMSolverMultistepInverseScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--damo-vilab--text-to-video-ms-1.7b/snapshots/8227dddca75a8561bf858d604cc5dae52b954d01/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


In [18]:
prompt = "spiderman is flying"

# Generate frames
result = pipe(prompt, num_inference_steps=25)
video_frames = result.frames

# Fix frames
video_frames = [(f[0][:, :, :3] * 255).astype("uint8") for f in video_frames]

# Export video
video_path = export_to_video(video_frames)

video_name = video_path.replace("/tmp", "")
print("Name:", video_name)

torch.cuda.empty_cache()

  0%|          | 0/25 [00:00<?, ?it/s]

Name: 4eiwgkwu.mp4


In [8]:
#prompt="dog is flying"
#video_frames =pipe(prompt, num_inference_steps=25).frames
#video_frames = [(f[0][:,:,:3]*255).astype("uint8") for f in video_frames]
#video_path =export_to_video(video_frames)
#video_name =video_path.replace("/tmp","")
#print("Name:",video_name)
#torch.cuda.empty_cache()

In [15]:
prompt = "dog is flying"

# Generate more frames (5 sec video)
result = pipe(
    prompt,
    num_inference_steps=25,
    num_frames=40   # 👈 5 sec at 8 FPS
)

video_frames = result.frames

# Fix frames
fixed_frames = []
for f in video_frames:
    if f.ndim == 4:
        f = f[0]  # remove batch dimension
    f = f[:, :, :3]
    f = (f * 255).astype("uint8")
    fixed_frames.append(f)

# Export video
video_path = export_to_video(fixed_frames, fps=8)

video_name = video_path.replace("/tmp", "")
print("Name:", video_name)

torch.cuda.empty_cache()

  0%|          | 0/25 [00:00<?, ?it/s]

Name: gvyoiddv.mp4
